In [ ]:
#############################################
"""
1. This file Generate the Dataset for training
"""

In [ ]:
#############################################
# 1. LoRa 데이터셋 생성용 Notebook
# - Clean IQ 및 (옵션) Clean Spectrogram 저장
#############################################

import numpy as np
import matplotlib.pyplot as plt
import os, sys

# 상위 디렉토리의 utils 모듈 사용
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa
import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import create_spectrogram_npy_dual  # 이미 쓰던 함수라 가정

print("Loaded my_lora_utils from:", utils.my_lora_utils.__file__)

In [ ]:
# LoRa 파라미터 설정
sf = 9                      # Spreading Factor
bw = 250_000               # 250 kHz (최소 250kHz로 증가)
OSF = 4                    # Oversampling Factor
fs = int(bw * OSF)         # 여기서는 250k * 4 = 1 MHz

symbol_time = 2**sf / bw   # 한 심볼 시간
N = 2**sf                  # 가능한 심볼 개수 (0 ~ 511)

print(f"SF = {sf}")
print(f"BW = {bw/1e3:.1f} kHz")
print(f"fs = {fs/1e6:.3f} MHz (OSF = {fs/bw:.1f})")
print(f"Symbol time = {symbol_time*1e3:.3f} ms")

# LoRa 심볼 생성 클래스 초기화
lora = LoRa(sf, bw)

In [ ]:
# 데이터 저장 폴더 설정
base_dir = "dataset_sf9_bw250k"

iq_dir = os.path.join(base_dir, "clean_iq")
spec_real_dir = os.path.join(base_dir, "clean_spec_real")
spec_imag_dir = os.path.join(base_dir, "clean_spec_imag")

os.makedirs(iq_dir, exist_ok=True)
os.makedirs(spec_real_dir, exist_ok=True)
os.makedirs(spec_imag_dir, exist_ok=True)

print("Created/Checked folders:")
print(" -", iq_dir)
print(" -", spec_real_dir)
print(" -", spec_imag_dir)

In [ ]:
# 테스트: 임의 코드워드 하나 생성해서 시간/스펙트럼 확인

test_sym = 10  # 아무거나
x = lora.gen_symbol_fs(test_sym, sf=sf, bw=bw, Fs=fs)  # time-domain complex IQ

print("x shape:", x.shape, "dtype:", x.dtype)

plt.figure(figsize=(10,3))
plt.plot(np.real(x), label="Real")
plt.plot(np.imag(x), label="Imag", alpha=0.7)
plt.title(f"Time-domain LoRa symbol (code={test_sym})")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 모든 심볼(0 ~ N-1)에 대해 clean IQ 및 clean spectrogram 저장
# 필요에 따라 num_symbols를 줄여서 테스트 가능

SAVE_SPEC = True   # 스펙트로그램도 저장할지 여부

num_symbols = N    # 2**sf = 512
print(f"Generating clean dataset for {num_symbols} symbols...")

for sym in range(num_symbols):
    # 1) 클린 심볼 IQ 생성
    x_clean = lora.gen_symbol_fs(sym, sf=sf, bw=bw, Fs=fs)  # complex numpy array

    # 2) 클린 IQ를 .npy로 저장 (real/imag 그대로)
    iq_path = os.path.join(iq_dir, f"sym_{sym:03d}_iq.npy")
    np.save(iq_path, x_clean.astype(np.complex64))

    # 3) (옵션) 스펙트로그램도 저장
    if SAVE_SPEC:
        # create_spectrogram_npy_dual(x, fs, snr, symbol_idx, label, folder_real, folder_imag)
        # 여기서는 snr=999(클린 표시), label=1 같은 고정값 사용
        create_spectrogram_npy_dual(
            x_clean,
            fs,
            999,          # snr 자리: clean이라는 의미로 999 사용
            sym,          # 심볼 인덱스
            1,            # label (필요 없으면 고정값)
            spec_real_dir,
            spec_imag_dir
        )

    if sym % 50 == 0:
        print(f"  - Generated symbol {sym}/{num_symbols-1}")

print("✅ Clean dataset generation done.")